In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [84]:
PJME_data = pd.read_csv("PJME_phase2_preprocessed.csv",parse_dates=["Datetime"])

In [85]:
PJME_data = PJME_data.set_index("Datetime")

# Sort chronologically
PJME_data = PJME_data.sort_index()

print(PJME_data.shape)
print(PJME_data.head())

(145224, 15)
                     PJME_MW  Hour  Day  Week  Month  DayOfWeek  IsWeekend  \
Datetime                                                                     
2002-01-08 01:00:00  29445.0     1    8     2      1          1          0   
2002-01-08 02:00:00  28670.0     2    8     2      1          1          0   
2002-01-08 03:00:00  28375.0     3    8     2      1          1          0   
2002-01-08 04:00:00  28542.0     4    8     2      1          1          0   
2002-01-08 05:00:00  29261.0     5    8     2      1          1          0   

                       Lag_1   Lag_24   Lag_48  Lag_168  Rolling_Mean_24  \
Datetime                                                                   
2002-01-08 01:00:00  31187.0  26862.0  27100.0  30393.0     33452.583333   
2002-01-08 02:00:00  29445.0  25976.0  26097.0  29265.0     33560.208333   
2002-01-08 03:00:00  28670.0  25641.0  25793.0  28357.0     33672.458333   
2002-01-08 04:00:00  28375.0  25666.0  25657.0  27899.0     

In [7]:
# Make sure data is chronological
PJME_data = PJME_data.sort_index()

n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = PJME_data.iloc[:train_end].copy()
validation = PJME_data.iloc[train_end:val_end].copy()
test = PJME_data.iloc[val_end:].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)


print(train.index.min(), "to", train.index.max())
print(validation.index.min(), "to", validation.index.max())
print(test.index.min(), "to", test.index.max())

Train: (101656, 15)
Validation: (21784, 15)
Test: (21784, 15)
2002-01-08 01:00:00 to 2013-08-13 16:00:00
2013-08-13 17:00:00 to 2016-02-07 08:00:00
2016-02-07 09:00:00 to 2018-08-03 00:00:00


In [8]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_48",
    "Lag_168",
    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

target = "PJME_MW"

X_train = train[features].copy()
X_val = validation[features].copy()
X_test = test[features].copy()

y_train = train[target].copy()
y_val = validation[target].copy()
y_test = test[target].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (101656, 15)
X_val: (21784, 15)
X_test: (21784, 15)


In [9]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_val_scaled = target_scaler.transform(y_val.values.reshape(-1, 1))
y_test_scaled = target_scaler.transform(y_test.values.reshape(-1, 1))

print("Scaling completed.")

Scaling completed.


In [10]:
def create_sequences(X, y, input_steps=24, output_steps=24):

    X_seq = []
    y_seq = []

    for i in range(input_steps, len(X) - output_steps + 1):

        X_seq.append(X[i-input_steps:i])
        y_seq.append(y[i:i+output_steps].flatten())

    return np.array(X_seq), np.array(y_seq)


X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=24,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=24,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=24,
    output_steps=24
)


print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101609, 24, 15)
y_train: (101609, 24)
X_val: (21737, 24, 15)
y_val: (21737, 24)
X_test: (21737, 24, 15)
y_test: (21737, 24)


In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [12]:
gru_model_es = Sequential([
    GRU(64, input_shape=(24, 15), return_sequences=False),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_model_es.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_model_es.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,272 (83.09 KB)

 Trainable params: 21,272 (83.09 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [14]:
gru_history_es = gru_model_es.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - loss: 0.1199 - mae: 0.2463 - val_loss: 0.0885 - val_mae: 0.2165
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0719 - mae: 0.1923 - val_loss: 0.0831 - val_mae: 0.2060
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0664 - mae: 0.1828 - val_loss: 0.0785 - val_mae: 0.2008
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0632 - mae: 0.1777 - val_loss: 0.0775 - val_mae: 0.1966
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0612 - mae: 0.1743 - val_loss: 0.0767 - val_mae: 0.1954
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0591 - mae: 0.1708 - val_loss: 0.0758 - val_mae: 0.1949
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0576 - mae: 0.1686 - val_loss: 0.0752 - val_mae: 0.1915
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0561 - mae: 0.1661 - val_loss: 0.0761 - val_mae: 0.1953
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [15]:
gru_model_es_pred_scaled = gru_model_es.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [16]:
gru_pred = target_scaler.inverse_transform(
    gru_model_es_pred_scaled.reshape(-1, 1)
).reshape(gru_model_es_pred_scaled.shape)

gru_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [17]:
from sklearn.metrics import ( mean_absolute_error, mean_squared_error, r2_score)

gru_mae = mean_absolute_error(gru_actual.flatten(), gru_pred.flatten())
gru_rmse = np.sqrt(mean_squared_error(gru_actual.flatten(), gru_pred.flatten()))
gru_mape = np.mean(np.abs((gru_actual.flatten() - gru_pred.flatten()) / gru_actual.flatten())) * 100
gru_r2 = r2_score(gru_actual.flatten(), gru_pred.flatten())
gru_bias = np.mean( gru_pred.flatten() - gru_actual.flatten())

print("GRU + Earlystopping")
print("MAE :", gru_mae)
print("RMSE:", gru_rmse)
print("MAPE:", gru_mape)
print("R²  :", gru_r2)
print("Bias:", gru_bias)

GRU + Earlystopping
MAE : 1375.318933922191
RMSE: 1951.2726023708944
MAPE: 4.30871507300687
R²  : 0.9082616898703646
Bias: -55.75306440643162


In [18]:
print("Epochs completed:", len(gru_history_es.history["loss"]))

Epochs completed: 13


**Dropout**

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout

gru_dropout_model = Sequential([
    GRU(64, input_shape=(24, 15), return_sequences=False),
    Dropout(0.2),
    Dense(64, activation="relu"),
    Dropout(0.2),
   Dense(24)
])

gru_dropout_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_dropout_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_1 (GRU)                     │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,272 (83.09 KB)

 Trainable params: 21,272 (83.09 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
gru_history_dropout = gru_dropout_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - loss: 0.1867 - mae: 0.3240 - val_loss: 0.0973 - val_mae: 0.2354
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.1210 - mae: 0.2631 - val_loss: 0.0894 - val_mae: 0.2212
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 9ms/step - loss: 0.1136 - mae: 0.2539 - val_loss: 0.0899 - val_mae: 0.2177
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1100 - mae: 0.2495 - val_loss: 0.0822 - val_mae: 0.2092
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.1077 - mae: 0.2469 - val_loss: 0.0830 - val_mae: 0.2098
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.1066 - mae: 0.2447 - val_loss: 0.0832 - val_mae: 0.2128
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.1044 - mae: 0.2428 - val_loss: 0.0806 - val_mae: 0.2058
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.1042 - mae: 0.2424 - val_loss: 0.0808 - val_mae: 0.2058
Epoch 9/15
1588/1588 ━━━━━━━

In [21]:
dropout_pred_scaled = gru_dropout_model.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [22]:
dropout_pred = target_scaler.inverse_transform(
    dropout_pred_scaled.reshape(-1, 1)
).reshape(dropout_pred_scaled.shape)

dropout_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [23]:
mae = mean_absolute_error(dropout_actual.flatten(), dropout_pred.flatten())
rmse = np.sqrt(mean_squared_error(dropout_actual.flatten(), dropout_pred.flatten()))
mape = np.mean(
    np.abs(
        (dropout_actual.flatten() - dropout_pred.flatten())
        / dropout_actual.flatten())) * 100
r2 = r2_score(dropout_actual.flatten(), dropout_pred.flatten())
bias = np.mean(dropout_pred.flatten() - dropout_actual.flatten())

print("== GRU + DROPOUT ==")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

== GRU + DROPOUT ==
MAE : 1456.0862753718159
RMSE: 1997.9895400434132
MAPE: 4.656728403462719
R²  : 0.9038163479466459
Bias: 262.8571468700222


**Batch Normalization**

In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import  BatchNormalization

gru_bn_model = Sequential([
    GRU(64, input_shape=(24, 15)),
    BatchNormalization(),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(24)
])

gru_bn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_bn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_2 (GRU)                     │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,784 (85.09 KB)

 Trainable params: 21,528 (84.09 KB)

 Non-trainable params: 256 (1.00 KB)

In [25]:
history_bn = gru_bn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.1774 - mae: 0.3078 - val_loss: 0.0979 - val_mae: 0.2296
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0990 - mae: 0.2367 - val_loss: 0.0901 - val_mae: 0.2189
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0898 - mae: 0.2245 - val_loss: 0.0846 - val_mae: 0.2075
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0853 - mae: 0.2179 - val_loss: 0.0811 - val_mae: 0.2072
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0818 - mae: 0.2131 - val_loss: 0.0763 - val_mae: 0.1992
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0794 - mae: 0.2099 - val_loss: 0.0814 - val_mae: 0.2001
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0768 - mae: 0.2064 - val_loss: 0.0764 - val_mae: 0.1964
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0757 - mae: 0.2047 - val_loss: 0.0813 - val_mae: 0.2016
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [26]:
bn_pred_scaled = gru_bn_model.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [27]:
rnn_bn_pred = target_scaler.inverse_transform(
    bn_pred_scaled.reshape(-1, 1)
).reshape(bn_pred_scaled.shape)

rnn_bn_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [28]:
mae = mean_absolute_error(rnn_bn_actual.flatten(), rnn_bn_pred.flatten())
rmse = np.sqrt(mean_squared_error(rnn_bn_actual.flatten(), rnn_bn_pred.flatten()))
mape = np.mean(np.abs((rnn_bn_actual.flatten() - rnn_bn_pred.flatten()) / rnn_bn_actual.flatten())) * 100
r2 = r2_score(rnn_bn_actual.flatten(), rnn_bn_pred.flatten())
bias = np.mean(rnn_bn_pred.flatten() - rnn_bn_actual.flatten())


print("== GRU + Batch ==")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

== GRU + Batch ==
MAE : 1483.330400916562
RMSE: 2089.898056513296
MAPE: 4.708198544061048
R²  : 0.8947638268795264
Bias: 331.9415656521774


**RMSPROP**

In [29]:
from tensorflow.keras.optimizers import RMSprop

gru_rmsprop = Sequential([
    GRU(64, input_shape=(24, 15), return_sequences=False),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_rmsprop.compile(
    optimizer=RMSprop(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

gru_rmsprop.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_3 (GRU)                     │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,272 (83.09 KB)

 Trainable params: 21,272 (83.09 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
history_rmsprop = gru_rmsprop.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.1207 - mae: 0.2509 - val_loss: 0.0931 - val_mae: 0.2236
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0786 - mae: 0.2023 - val_loss: 0.0873 - val_mae: 0.2129
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0713 - mae: 0.1903 - val_loss: 0.0867 - val_mae: 0.2089
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0672 - mae: 0.1836 - val_loss: 0.0770 - val_mae: 0.1971
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0644 - mae: 0.1790 - val_loss: 0.0812 - val_mae: 0.2052
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0622 - mae: 0.1755 - val_loss: 0.0836 - val_mae: 0.2083
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0603 - mae: 0.1724 - val_loss: 0.0845 - val_mae: 0.2093
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0587 - mae: 0.1700 - val_loss: 0.0775 - val_mae: 0.1954
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [31]:
rmsprop_pred_scaled = gru_rmsprop.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", rmsprop_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Prediction shape: (21737, 24)


In [32]:
rmsprop_pred = target_scaler.inverse_transform(
    rmsprop_pred_scaled.reshape(-1, 1)
).reshape(rmsprop_pred_scaled.shape)

rmsprop_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [33]:
rmsprop_mae = mean_absolute_error(rmsprop_actual.flatten(), rmsprop_pred.flatten())
rmsprop_rmse = np.sqrt(mean_squared_error(rmsprop_actual.flatten(), rmsprop_pred.flatten()))
rmsprop_mape = np.mean(np.abs((rmsprop_actual.flatten() - rmsprop_pred.flatten())/ rmsprop_actual.flatten()) * 100
rmsprop_r2 = r2_score(rmsprop_actual.flatten(), rmsprop_pred.flatten())
rmsprop_bias = np.mean(rmsprop_pred.flatten() - rmsprop_actual.flatten())

print("=== GRU + RMSPROP ===")
print("MAE :", rmsprop_mae)
print("RMSE:", rmsprop_rmse)
print("MAPE:", rmsprop_mape)
print("R²  :", rmsprop_r2)
print("Bias:", rmsprop_bias)

=== GRU + RMSPROP ===
MAE : 1407.4961324689637
RMSE: 2014.8363911184438
MAPE: 4.399045300530449
R²  : 0.9021874874137391
Bias: -51.518069700843704


**SGD**

In [34]:
from tensorflow.keras.optimizers import SGD

gru_sgd = Sequential([
    GRU(64, input_shape=(24, 15), return_sequences=False),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_sgd.compile(
    optimizer=SGD(learning_rate=0.001,momentum=0.9),
    loss="mse",
    metrics=["mae"]
)

gru_sgd.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_4 (GRU)                     │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,272 (83.09 KB)

 Trainable params: 21,272 (83.09 KB)

 Non-trainable params: 0 (0.00 B)

In [35]:
history_sgd = gru_sgd.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.5470 - mae: 0.5717 - val_loss: 0.3354 - val_mae: 0.4600
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.2608 - mae: 0.4044 - val_loss: 0.2066 - val_mae: 0.3593
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.1828 - mae: 0.3357 - val_loss: 0.1735 - val_mae: 0.3251
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.1571 - mae: 0.3079 - val_loss: 0.1562 - val_mae: 0.3058
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.1428 - mae: 0.2911 - val_loss: 0.1458 - val_mae: 0.2931
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.1337 - mae: 0.2799 - val_loss: 0.1387 - val_mae: 0.2845
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.1272 - mae: 0.2716 - val_loss: 0.1338 - val_mae: 0.2779
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.1222 - mae: 0.2650 - val_loss: 0.1298 - val_mae: 0.2732
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━━━━━

In [36]:
sgd_pred_scaled = gru_sgd.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", sgd_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Prediction shape: (21737, 24)


In [37]:
sgd_pred = target_scaler.inverse_transform(
    sgd_pred_scaled.reshape(-1, 1)
).reshape(sgd_pred_scaled.shape)

sgd_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [38]:
sgd_mae = mean_absolute_error(sgd_actual.flatten(), sgd_pred.flatten())
sgd_rmse = np.sqrt(mean_squared_error(sgd_actual.flatten(), sgd_pred.flatten()))
sgd_mape = np.mean(np.abs((sgd_actual.flatten() - sgd_pred.flatten()) / sgd_actual.flatten())) * 100
sgd_r2 = r2_score(sgd_actual.flatten(), sgd_pred.flatten())
sgd_bias = np.mean(sgd_pred.flatten() - sgd_actual.flatten())

print("=== GRU + SGD ===")
print("MAE :", sgd_mae)
print("RMSE:", sgd_rmse)
print("MAPE:", sgd_mape)
print("R²  :", sgd_r2)
print("Bias:", sgd_bias)

=== GRU + SGD ===
MAE : 1703.7519988150245
RMSE: 2292.5344193121578
MAPE: 5.456449961834007
R²  : 0.8733670943755459
Bias: 133.18358558839287


Learning Rate Scheduling

In [39]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import RMSprop

gru_lr = Sequential([
    GRU(64, input_shape=(24, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_lr.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [40]:
history_lr = gru_lr.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    callbacks=[lr_scheduler],
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.1203 - mae: 0.2500 - val_loss: 0.0949 - val_mae: 0.2238 - learning_rate: 0.0010
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0766 - mae: 0.1993 - val_loss: 0.0863 - val_mae: 0.2113 - learning_rate: 0.0010
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0698 - mae: 0.1882 - val_loss: 0.0816 - val_mae: 0.2049 - learning_rate: 0.0010
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0660 - mae: 0.1819 - val_loss: 0.0796 - val_mae: 0.1999 - learning_rate: 0.0010
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0633 - mae: 0.1777 - val_loss: 0.0825 - val_mae: 0.2073 - learning_rate: 0.0010
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0611 - mae: 0.1741 - val_loss: 0.0753 - val_mae: 0.1979 - learning_rate: 0.0010
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0594 - mae: 0.1713 - val_loss: 0.0772 - val_mae: 0.1985 - learni

In [41]:
lr_pred_scaled = gru_lr.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", lr_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21737, 24)


In [42]:
lr_pred = target_scaler.inverse_transform(
    lr_pred_scaled.reshape(-1, 1)
).reshape(lr_pred_scaled.shape)

lr_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [43]:
lr_mae = mean_absolute_error(lr_actual.flatten(), lr_pred.flatten())
lr_rmse = np.sqrt(mean_squared_error(lr_actual.flatten(),lr_pred.flatten()))
lr_mape = np.mean(np.abs((lr_actual.flatten() - lr_pred.flatten()) / lr_actual.flatten())) * 100
lr_r2 = r2_score(lr_actual.flatten(), lr_pred.flatten())
lr_bias = np.mean(lr_pred.flatten() - lr_actual.flatten())

print("==== GRU + RMSPROP + LearningRate ====")
print("MAE :", lr_mae)
print("RMSE:", lr_rmse)
print("MAPE:", lr_mape)
print("R²  :", lr_r2)
print("Bias:", lr_bias)

==== GRU + RMSPROP + LearningRate ====
MAE : 1375.9925618931359
RMSE: 1957.2240205840515
MAPE: 4.3532224496727885
R²  : 0.9077012293143857
Bias: 207.08011843135887


**Layers**

In [44]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense

gru_deep = Sequential([
    GRU(64, return_sequences=True, input_shape=(24, 15)),
    GRU(32, return_sequences=False),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_deep.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_deep.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_6 (GRU)                     │ (None, 24, 64)         │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_7 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,632 (111.84 KB)

 Trainable params: 28,632 (111.84 KB)

 Non-trainable params: 0 (0.00 B)

In [45]:
history_deep = gru_deep.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.1192 - mae: 0.2455 - val_loss: 0.0901 - val_mae: 0.2195
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 24s 11ms/step - loss: 0.0715 - mae: 0.1922 - val_loss: 0.0860 - val_mae: 0.2108
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.0656 - mae: 0.1824 - val_loss: 0.0823 - val_mae: 0.2087
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 30s 19ms/step - loss: 0.0621 - mae: 0.1768 - val_loss: 0.0771 - val_mae: 0.1994
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 29s 18ms/step - loss: 0.0594 - mae: 0.1724 - val_loss: 0.0804 - val_mae: 0.2019
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 39s 17ms/step - loss: 0.0572 - mae: 0.1689 - val_loss: 0.0780 - val_mae: 0.1939
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0553 - mae: 0.1660 - val_loss: 0.0795 - val_mae: 0.1999
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0535 - mae: 0.1633 - val_loss: 0.0793 - val_mae: 0.2009
Epoch 9/15
1588/1588 ━━━━

In [46]:
deep_pred_scaled = gru_deep.predict(
    X_test_seq,
    batch_size=64
)
print("Prediction shape:", deep_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21737, 24)


In [47]:
deep_pred = target_scaler.inverse_transform(
    deep_pred_scaled.reshape(-1, 1)
).reshape(deep_pred_scaled.shape)

deep_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [48]:
deep_mae = mean_absolute_error(deep_actual.flatten(), deep_pred.flatten())
deep_rmse = np.sqrt(mean_squared_error(deep_actual.flatten(), deep_pred.flatten()))
deep_mape = np.mean(np.abs((deep_actual.flatten() - deep_pred.flatten()) / deep_actual.flatten())) * 100
deep_r2 = r2_score(deep_actual.flatten(), deep_pred.flatten())
deep_bias = np.mean(deep_pred.flatten() - deep_actual.flatten())


print("==== GRU + Layers ==========")
print("MAE :", deep_mae)
print("RMSE:", deep_rmse)
print("MAPE:", deep_mape)
print("R²  :", deep_r2)
print("Bias:", deep_bias)

==== GRU + Layers ==========
MAE : 1453.0291605274908
RMSE: 2110.155354698644
MAPE: 4.560434647134552
R²  : 0.8927138395504848
Bias: 233.1781631925775


**Neurons**

In [49]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense

gru_128 = Sequential([
    GRU(128, input_shape=(24, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_128.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_8 (GRU)                     │ (None, 128)            │        55,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65,496 (255.84 KB)

 Trainable params: 65,496 (255.84 KB)

 Non-trainable params: 0 (0.00 B)

In [50]:
history_128 = gru_128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.1077 - mae: 0.2339 - val_loss: 0.0877 - val_mae: 0.2137
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0695 - mae: 0.1882 - val_loss: 0.0824 - val_mae: 0.2095
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0636 - mae: 0.1783 - val_loss: 0.0778 - val_mae: 0.1992
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0599 - mae: 0.1723 - val_loss: 0.0782 - val_mae: 0.1974
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0571 - mae: 0.1676 - val_loss: 0.0758 - val_mae: 0.1959
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0547 - mae: 0.1640 - val_loss: 0.0762 - val_mae: 0.1939
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0522 - mae: 0.1601 - val_loss: 0.0729 - val_mae: 0.1889
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.0495 - mae: 0.1566 - val_loss: 0.0767 - val_mae: 0.1938
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [51]:
gru_128_pred_scaled = gru_128.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", gru_128_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Prediction shape: (21737, 24)


In [52]:
gru128_pred = target_scaler.inverse_transform(
    gru_128_pred_scaled.reshape(-1, 1)
).reshape(gru_128_pred_scaled.shape)

gru128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [53]:
rnn128_mae = mean_absolute_error(gru128_actual.flatten(), gru128_pred.flatten())
rnn128_rmse = np.sqrt(mean_squared_error(gru128_actual.flatten(), gru128_pred.flatten()))
rnn128_mape = np.mean(np.abs((gru128_actual.flatten() - gru128_pred.flatten()) / gru128_actual.flatten())) * 100
rnn128_r2 = r2_score(gru128_actual.flatten(), gru128_pred.flatten())
rnn128_bias = np.mean(gru128_pred.flatten() - gru128_actual.flatten())

print("=== GRU 128 NEURONS ===")
print("MAE :", rnn128_mae)
print("RMSE:", rnn128_rmse)
print("MAPE:", rnn128_mape)
print("R²  :", rnn128_r2)
print("Bias:", rnn128_bias)

=== GRU 128 NEURONS ===
MAE : 1474.310833792929
RMSE: 2146.801689733654
MAPE: 4.603648906366726
R²  : 0.8889550790170824
Bias: 83.93457800612507


32 **neurons**

In [54]:
gru_32 = Sequential([
    GRU(32, input_shape=(24, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_32.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_9 (GRU)                     │ (None, 32)             │         4,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,376 (32.72 KB)

 Trainable params: 8,376 (32.72 KB)

 Non-trainable params: 0 (0.00 B)

In [55]:
history_32 = gru_32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - loss: 0.1406 - mae: 0.2670 - val_loss: 0.0967 - val_mae: 0.2254
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.0776 - mae: 0.2012 - val_loss: 0.0860 - val_mae: 0.2097
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0711 - mae: 0.1904 - val_loss: 0.0835 - val_mae: 0.2062
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0676 - mae: 0.1845 - val_loss: 0.0791 - val_mae: 0.1998
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0654 - mae: 0.1808 - val_loss: 0.0817 - val_mae: 0.2004
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0640 - mae: 0.1785 - val_loss: 0.0780 - val_mae: 0.1970
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.0624 - mae: 0.1756 - val_loss: 0.0773 - val_mae: 0.1967
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0615 - mae: 0.1742 - val_loss: 0.0802 - val_mae: 0.1984
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [56]:
gru32_pred_scaled = gru_32.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", gru32_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21737, 24)


In [57]:
gru32_pred = target_scaler.inverse_transform(
    gru32_pred_scaled.reshape(-1, 1)
).reshape(gru32_pred_scaled.shape)

gru32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [60]:
gru32_mae = mean_absolute_error(gru32_actual.flatten(), gru32_pred.flatten())
gru32_rmse = np.sqrt(mean_squared_error(gru32_actual.flatten(), gru32_pred.flatten()))
gru32_mape = np.mean(np.abs((gru32_actual.flatten() - gru32_pred.flatten()) / gru32_actual.flatten())) * 100
gru32_r2 = r2_score(gru32_actual.flatten(), gru32_pred.flatten())
gru32_bias = np.mean(gru32_pred.flatten() - gru32_actual.flatten())

print("=== GRU 32 NEURONS ===")
print("MAE :", gru32_mae)
print("RMSE:", gru32_rmse)
print("MAPE:", gru32_mape)
print("R²  :", gru32_r2)
print("Bias:", gru32_bias)

=== GRU 32 NEURONS ===
MAE : 1448.363232995434
RMSE: 2007.45902016853
MAPE: 4.603016084717212
R²  : 0.9029024617072801
Bias: 317.8527198945119


Batch Size(32)

In [61]:
gru_batch32 = Sequential([
    GRU(64, input_shape=(48, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_batch32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_batch32 = gru_batch32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=32,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 35s 10ms/step - loss: 0.1044 - mae: 0.2310 - val_loss: 0.0887 - val_mae: 0.2142
Epoch 2/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - loss: 0.0704 - mae: 0.1896 - val_loss: 0.0830 - val_mae: 0.2045
Epoch 3/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 0.0652 - mae: 0.1809 - val_loss: 0.0753 - val_mae: 0.1954
Epoch 4/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 21s 6ms/step - loss: 0.0618 - mae: 0.1753 - val_loss: 0.0787 - val_mae: 0.2023
Epoch 5/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 21s 6ms/step - loss: 0.0591 - mae: 0.1709 - val_loss: 0.0768 - val_mae: 0.1930
Epoch 6/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 19s 6ms/step - loss: 0.0571 - mae: 0.1678 - val_loss: 0.0756 - val_mae: 0.1936
Epoch 7/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 0.0551 - mae: 0.1649 - val_loss: 0.0808 - val_mae: 0.2002
Epoch 8/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 0.0532 - mae: 0.1623 - val_loss: 0.0763 - val_mae: 0.1958
Epoch 9/15
3176/3176 ━━━━━━━━━━

In [62]:
gru_batch32_pred_scaled = gru_batch32.predict(
    X_test_seq,
    batch_size=32
)

print("Prediction shape:", gru_batch32_pred_scaled.shape)

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
Prediction shape: (21737, 24)


In [63]:
grubatch32_pred = target_scaler.inverse_transform(
    gru_batch32_pred_scaled.reshape(-1, 1)
).reshape(gru_batch32_pred_scaled.shape)

grubatch32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [65]:
grub32_mae = mean_absolute_error(grubatch32_actual.flatten(), grubatch32_pred.flatten())
grub32_rmse = np.sqrt(mean_squared_error(grubatch32_actual.flatten(), grubatch32_pred.flatten()))
grub32_mape = np.mean(np.abs((grubatch32_actual.flatten() - grubatch32_pred.flatten()) / grubatch32_actual.flatten())) * 100
grub32_r2 = r2_score(grubatch32_actual.flatten(), grubatch32_pred.flatten())
gru32_bias = np.mean(grubatch32_pred.flatten() - grubatch32_actual.flatten())

print("=== GRU batch32 NEURONS ===")
print("MAE :", grub32_mae)
print("RMSE:", grub32_rmse)
print("MAPE:", grub32_mape)
print("R²  :", grub32_r2)
print("Bias:", gru32_bias)

=== GRU batch32 NEURONS ===
MAE : 1460.2804909293545
RMSE: 2085.897551494003
MAPE: 4.581074732721486
R²  : 0.8951663296697514
Bias: 96.32098776816196


BatchSize(128)

In [66]:
gru_batch128 = Sequential([
    GRU(64, input_shape=(48, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_batch128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_batch128 = gru_batch128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=128,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 0.1460 - mae: 0.2710 - val_loss: 0.0930 - val_mae: 0.2208
Epoch 2/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 0.0755 - mae: 0.1987 - val_loss: 0.0858 - val_mae: 0.2101
Epoch 3/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.0689 - mae: 0.1876 - val_loss: 0.0828 - val_mae: 0.2072
Epoch 4/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.0653 - mae: 0.1815 - val_loss: 0.0795 - val_mae: 0.2023
Epoch 5/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.0629 - mae: 0.1775 - val_loss: 0.0803 - val_mae: 0.2007
Epoch 6/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.0610 - mae: 0.1744 - val_loss: 0.0770 - val_mae: 0.1956
Epoch 7/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 0.0594 - mae: 0.1717 - val_loss: 0.0765 - val_mae: 0.1944
Epoch 8/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.0583 - mae: 0.1699 - val_loss: 0.0768 - val_mae: 0.1992
Epoch 9/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - lo

In [67]:
gru_batch128_pred_scaled = gru_batch128.predict(
    X_test_seq,
    batch_size=128
)

print("Prediction shape:", gru_batch128_pred_scaled.shape)

170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21737, 24)


In [68]:
batch128_pred = target_scaler.inverse_transform(
    gru_batch128_pred_scaled.reshape(-1, 1)
).reshape(gru_batch128_pred_scaled.shape)

batch128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [73]:
grub128_mae = mean_absolute_error(batch128_actual.flatten(), batch128_pred.flatten())
grub128_rmse = np.sqrt(mean_squared_error(batch128_actual.flatten(), batch128_pred.flatten()))
grub128_mape = np.mean(np.abs((batch128_actual.flatten() - batch128_pred.flatten()) / batch128_actual.flatten())) * 100
grub128_r2 = r2_score(batch128_actual.flatten(), batch128_pred.flatten())
grub128_bias = np.mean(batch128_pred.flatten() - batch128_actual.flatten())

print("=== GRU batch128  ===")
print("MAE :", grub128_mae)
print("RMSE:", grub128_rmse)
print("MAPE:", grub128_mape)
print("R²  :", grub128_r2)
print("Bias:", grub128_bias)

=== GRU batch128  ===
MAE : 1397.917115233087
RMSE: 1997.6426046591107
MAPE: 4.4265580938401765
R²  : 0.9038497481366283
Bias: 235.6968148347288


**HyperParameter Tuning**

In [74]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.8 MB/s eta 0:00:00


In [75]:
import keras_tuner as kt

In [76]:
def build_gru(hp):

    model = Sequential()
    model.add(
        GRU(
            units=hp.Choice(
                "rnn_units",
                values=[32, 64, 128]
            ),
            input_shape=(24, 15)
        )
    )
    model.add(
        Dense(
            units=hp.Choice(
                "dense_units",
                values=[32, 64, 128]
            ),
            activation="relu"
        )
    )
    model.add(Dense(24))
    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0001, 0.0005, 0.001]
    )
    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [77]:
tuner = kt.RandomSearch(
    build_gru,
    objective="val_loss",
    max_trials=6,
    executions_per_trial=1,
    directory="hyperparameter_tuning",
    project_name="gru_24_to_24"
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [78]:
tuner.search(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Trial 6 Complete [00h 02m 47s]
val_loss: 0.076868437230587

Best val_loss So Far: 0.0715186819434166
Total elapsed time: 00h 17m 42s


In [79]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("BEST HYPERPARAMETERS")
print("RNN Units:",best_hp.get("rnn_units"))
print("Dense Units:",best_hp.get("dense_units"))
print("Learning Rate:",best_hp.get("learning_rate"))

BEST HYPERPARAMETERS
RNN Units: 64
Dense Units: 64
Learning Rate: 0.0005


In [80]:
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 16 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,272 (83.09 KB)

 Trainable params: 21,272 (83.09 KB)

 Non-trainable params: 0 (0.00 B)

In [81]:
history_best = best_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 26s 15ms/step - loss: 0.0532 - mae: 0.1617 - val_loss: 0.0746 - val_mae: 0.1892
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 27s 7ms/step - loss: 0.0526 - mae: 0.1607 - val_loss: 0.0759 - val_mae: 0.1943
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.0519 - mae: 0.1599 - val_loss: 0.0731 - val_mae: 0.1891
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0512 - mae: 0.1589 - val_loss: 0.0743 - val_mae: 0.1909
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0507 - mae: 0.1582 - val_loss: 0.0766 - val_mae: 0.1937
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0500 - mae: 0.1573 - val_loss: 0.0771 - val_mae: 0.1960
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0493 - mae: 0.1563 - val_loss: 0.0759 - val_mae: 0.1922
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0487 - mae: 0.1555 - val_loss: 0.0744 - val_mae: 0.1900
Epoch 9/15
1588/1588 ━━━━━━━━━━

In [82]:
best_pred_scaled = best_model.predict(
    X_test_seq,
    batch_size=64
)

best_pred = target_scaler.inverse_transform(
    best_pred_scaled.reshape(-1, 1)
).reshape(best_pred_scaled.shape)

best_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [83]:
best_mae = mean_absolute_error(best_actual.flatten(), best_pred.flatten())
best_rmse = np.sqrt(mean_squared_error(best_actual.flatten(), best_pred.flatten()))
best_mape = np.mean(np.abs((best_actual.flatten() - best_pred.flatten()) / best_actual.flatten())) * 100
best_r2 = r2_score(best_actual.flatten(), best_pred.flatten())
best_bias = np.mean(best_pred.flatten() - best_actual.flatten())

print("HYPERPARAMETER TUNED GRU")
print("MAE :", best_mae)
print("RMSE:", best_rmse)
print("MAPE:", best_mape)
print("R²  :", best_r2)
print("Bias:", best_bias)

HYPERPARAMETER TUNED GRU
MAE : 1417.0926513316208
RMSE: 2030.5244214899785
MAPE: 4.449079013679001
R²  : 0.9006583710810352
Bias: 153.85867687185038





**Additional Logs**

In [89]:
# Additional lag features

PJME_data["Lag_2"] = PJME_data["PJME_MW"].shift(2)
PJME_data["Lag_3"] = PJME_data["PJME_MW"].shift(3)
PJME_data["Lag_6"] = PJME_data["PJME_MW"].shift(6)
PJME_data["Lag_12"] = PJME_data["PJME_MW"].shift(12)
PJME_data["Lag_72"] = PJME_data["PJME_MW"].shift(72)
PJME_data["Lag_336"] = PJME_data["PJME_MW"].shift(336)

In [90]:
PJME_data = PJME_data.dropna().copy()

print("Missing values:")
print(PJME_data.isnull().sum().sum())

print("Shape:")
print(PJME_data.shape)

Missing values:
0
Shape:
(144552, 21)


In [91]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",

    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Lag_6",
    "Lag_12",
    "Lag_24",
    "Lag_48",
    "Lag_72",
    "Lag_168",
    "Lag_336",

    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

print("Number of features:", len(features))

Number of features: 21


In [92]:
n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_data = PJME_data.iloc[:train_end]
val_data = PJME_data.iloc[train_end:val_end]
test_data = PJME_data.iloc[val_end:]

print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

Train: (101186, 21)
Validation: (21683, 21)
Test: (21683, 21)


In [93]:
X_train = train_data[features]
y_train = train_data[["PJME_MW"]]

X_val = val_data[features]
y_val = val_data[["PJME_MW"]]

X_test = test_data[features]
y_test = test_data[["PJME_MW"]]

In [94]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(y_train)
y_val_scaled = target_scaler.transform(y_val)
y_test_scaled = target_scaler.transform(y_test)

print("Scaling completed.")

Scaling completed.


In [95]:
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=48,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=48,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=48,
    output_steps=24
)

print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101115, 48, 21)
y_train: (101115, 24)
X_val: (21612, 48, 21)
y_val: (21612, 24)
X_test: (21612, 48, 21)
y_test: (21612, 24)


In [96]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense

gru_lags = Sequential([
    GRU(64, input_shape=(48, 21)),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_lags.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_lags.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_1 (GRU)                     │ (None, 64)             │        16,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,424 (87.59 KB)

 Trainable params: 22,424 (87.59 KB)

 Non-trainable params: 0 (0.00 B)

In [97]:
history_lags = gru_lags.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.1160 - mae: 0.2423 - val_loss: 0.0918 - val_mae: 0.2194
Epoch 2/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0709 - mae: 0.1905 - val_loss: 0.0800 - val_mae: 0.2034
Epoch 3/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0647 - mae: 0.1803 - val_loss: 0.0794 - val_mae: 0.2034
Epoch 4/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - loss: 0.0610 - mae: 0.1742 - val_loss: 0.0764 - val_mae: 0.1979
Epoch 5/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0582 - mae: 0.1698 - val_loss: 0.0743 - val_mae: 0.1938
Epoch 6/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0561 - mae: 0.1664 - val_loss: 0.0724 - val_mae: 0.1914
Epoch 7/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0540 - mae: 0.1634 - val_loss: 0.0753 - val_mae: 0.1980
Epoch 8/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0521 - mae: 0.1609 - val_loss: 0.0744 - val_mae: 0.1926
Epoch 9/15
1580/1580 ━━━

In [98]:
pred_scaled = gru_lags.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", pred_scaled.shape)

338/338 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Prediction shape: (21612, 24)


In [99]:
pred = target_scaler.inverse_transform(
    pred_scaled.reshape(-1, 1)
).reshape(pred_scaled.shape)

actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [100]:
mae = mean_absolute_error(actual.flatten(), pred.flatten())
rmse = np.sqrt(mean_squared_error(actual.flatten(), pred.flatten()))
mape = np.mean(np.abs((actual.flatten() - pred.flatten()) / actual.flatten())) * 100
r2 = r2_score(actual.flatten(), pred.flatten())
bias = np.mean(pred.flatten() - actual.flatten())

print("=== GRU + ADDITIONAL LAGS ===")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

=== GRU + ADDITIONAL LAGS ===
MAE : 1526.825054437229
RMSE: 2220.623763640765
MAPE: 4.808713357082635
R²  : 0.881181101750546
Bias: 314.7786655269521


In [1]:
import pandas as pd
import os

gru_24_24_results = [
    ["GRU Baseline", 24, 24, 1429.7045531539566, 2086.4722667763826, 4.502954834750559, 0.8951085532823124, 241.54513742379305],
    ["GRU + EarlyStopping", 24, 24, 1375.318933922191, 1951.2726023708944, 4.30871507300687, 0.9082616898703646, -55.75306440643162],
    ["GRU + Dropout", 24, 24, 1456.0862753718159, 1997.9895400434132, 4.656728403462719, 0.9038163479466459, 262.8571468700222],
    ["GRU + Batch Normalization", 24, 24, 1483.330400916562, 2089.898056513296, 4.708198544061048, 0.8947638268795264, 331.9415656521774],
    ["GRU + RMSprop", 24, 24, 1407.4961324689637, 2014.8363911184438, 4.399045300530449, 0.9021874874137391, -51.518069700843704],
    ["GRU + SGD", 24, 24, 1703.7519988150245, 2292.5344193121578, 5.456449961834007, 0.8733670943755459, 133.18358558839287],
    ["GRU + RMSprop + Learning Rate", 24, 24, 1375.9925618931359, 1957.2240205840515, 4.3532224496727885, 0.9077012293143857, 207.08011843135887],
    ["GRU + Additional Layers", 24, 24, 1453.0291605274908, 2110.155354698644, 4.560434647134552, 0.8927138395504848, 233.1781631925775],
    ["GRU 128 Neurons", 24, 24, 1474.310833792929, 2146.801689733654, 4.603648906366726, 0.8889550790170824, 83.93457800612507],
    ["GRU 32 Neurons", 24, 24, 1448.363232995434, 2007.45902016853, 4.603016084717212, 0.9029024617072801, 317.8527198945119],
    ["GRU Batch Size 32", 24, 24, 1460.2804909293545, 2085.897551494003, 4.581074732721486, 0.8951663296697514, 96.32098776816196],
    ["GRU Batch Size 128", 24, 24, 1397.917115233087, 1997.6426046591107, 4.4265580938401765, 0.9038497481366283, 235.6968148347288],
    ["GRU Hyperparameter Tuned", 24, 24, 1417.0926513316208, 2030.5244214899785, 4.449079013679001, 0.9006583710810352, 153.85867687185038],
    ["GRU + Additional Lags", 24, 24, 1526.825054437229, 2220.623763640765, 4.808713357082635, 0.881181101750546, 314.7786655269521]
]

gru_24_24_df = pd.DataFrame(
    gru_24_24_results,
    columns=[
        "Model",
        "Input Hours",
        "Output Hours",
        "MAE",
        "RMSE",
        "MAPE (%)",
        "R²",
        "Bias"
    ]
)

# Display with 6 decimal places
display(
    gru_24_24_df.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "MAPE (%)": "{:.6f}",
        "R²": "{:.6f}",
        "Bias": "{:.6f}"
    })
)

# Save inside existing comparison folder
os.makedirs("comparison", exist_ok=True)

gru_24_24_df.to_csv(
    "comparison/GRU_24h_to_24h_results.csv",
    index=False,
    float_format="%.6f"
)

print("Saved: comparison/GRU_24h_to_24h_results.csv")

,Model,Input Hours,Output Hours,MAE,RMSE,MAPE (%),R²,Bias
0,GRU Baseline,24,24,1429.704553,2086.472267,4.502955,0.895109,241.545137
1,GRU + EarlyStopping,24,24,1375.318934,1951.272602,4.308715,0.908262,-55.753064
2,GRU + Dropout,24,24,1456.086275,1997.989540,4.656728,0.903816,262.857147
3,GRU + Batch Normalization,24,24,1483.330401,2089.898057,4.708199,0.894764,331.941566
4,GRU + RMSprop,24,24,1407.496132,2014.836391,4.399045,0.902187,-51.518070
5,GRU + SGD,24,24,1703.751999,2292.534419,5.456450,0.873367,133.183586
6,GRU + RMSprop + Learning Rate,24,24,1375.992562,1957.224021,4.353222,0.907701,207.080118
7,GRU + Additional Layers,24,24,1453.029161,2110.155355,4.560435,0.892714,233.178163
8,GRU 128 Neurons,24,24,1474.310834,2146.801690,4.603649,0.888955,83.934578
9,GRU 32 Neurons,24,24,1448.363233,2007.459020,4.603016,0.902902,317.852720


Saved: comparison/GRU_24h_to_24h_results.csv


In [2]:
best_gru_24_24 = gru_24_24_df.loc[gru_24_24_df["Model"] == "GRU + EarlyStopping"]
display(best_gru_24_24.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "MAPE (%)": "{:.6f}",
        "R²": "{:.6f}",
        "Bias": "{:.6f}"
    })
)

,Model,Input Hours,Output Hours,MAE,RMSE,MAPE (%),R²,Bias
1,GRU + EarlyStopping,24,24,1375.318934,1951.272602,4.308715,0.908262,-55.753064
